In [1]:
import pandas  as pd
import numpy as np


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures,StandardScaler,MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn import metrics,linear_model,tree,ensemble
import matplotlib.pyplot as plt

: 

In [3]:
%pip install machine-learning-datasets==0.1.23 --no-deps


Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

df = pd.read_csv(
    "https://media.githubusercontent.com/media/PacktPublishing/Interpretable-Machine-Learning-with-Python/refs/heads/master/datasets/aa-domestic-delays-2018.csv.zip",
    compression="zip"
)

In [5]:
df.head()

,FL_NUM,ORIGIN,DEST,PLANNED_DEP_DATETIME,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_AFPH,DEP_RFPH,TAXI_OUT,...,DISTANCE,CRS_ARR_TIME,ARR_AFPH,ARR_RFPH,ARR_DELAY,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,419,DCA,DFW,2018-01-01 11:55:00,1155,1149.0,-6.0,34.444444,0.956790,14.0,...,1192.0,1434,74.347826,0.854573,-14.0,0.0,0.0,0.0,0.0,0.0
1,419,DFW,DCA,2018-01-01 07:05:00,705,700.0,-5.0,17.454545,0.242424,16.0,...,1192.0,1056,30.731707,0.731707,-19.0,0.0,0.0,0.0,0.0,0.0
2,420,DEN,PHL,2018-01-01 11:48:00,1148,1145.0,-3.0,94.736842,0.947368,14.0,...,1558.0,1720,45.882353,1.092437,-9.0,0.0,0.0,0.0,0.0,0.0
3,420,PHL,DEN,2018-01-01 08:25:00,825,824.0,-1.0,33.559322,0.860495,16.0,...,1558.0,1056,74.594595,0.867379,-23.0,0.0,0.0,0.0,0.0,0.0
4,421,DCA,CLT,2018-01-01 11:55:00,1155,1147.0,-8.0,33.461538,0.929487,13.0,...,331.0,1334,90.612245,1.006803,-11.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 899527 entries, 0 to 899526
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   FL_NUM                899527 non-null  int64  
 1   ORIGIN                899527 non-null  str    
 2   DEST                  899527 non-null  str    
 3   PLANNED_DEP_DATETIME  899527 non-null  str    
 4   CRS_DEP_TIME          899527 non-null  int64  
 5   DEP_TIME              899527 non-null  float64
 6   DEP_DELAY             899527 non-null  float64
 7   DEP_AFPH              899527 non-null  float64
 8   DEP_RFPH              899527 non-null  float64
 9   TAXI_OUT              899527 non-null  float64
 10  WHEELS_OFF            899527 non-null  float64
 11  CRS_ELAPSED_TIME      899527 non-null  float64
 12  PCT_ELAPSED_TIME      899527 non-null  float64
 13  DISTANCE              899527 non-null  float64
 14  CRS_ARR_TIME          899527 non-null  int64  
 15  ARR_AFPH   

In [7]:
df.isnull().sum()

FL_NUM                  0
ORIGIN                  0
DEST                    0
PLANNED_DEP_DATETIME    0
CRS_DEP_TIME            0
DEP_TIME                0
DEP_DELAY               0
DEP_AFPH                0
DEP_RFPH                0
TAXI_OUT                0
WHEELS_OFF              0
CRS_ELAPSED_TIME        0
PCT_ELAPSED_TIME        0
DISTANCE                0
CRS_ARR_TIME            0
ARR_AFPH                0
ARR_RFPH                0
ARR_DELAY               0
CARRIER_DELAY           0
WEATHER_DELAY           0
NAS_DELAY               0
SECURITY_DELAY          0
LATE_AIRCRAFT_DELAY     0
dtype: int64

In [8]:
df['PLANNED_DEP_DATETIME']=pd.to_datetime(df['PLANNED_DEP_DATETIME'])
df['DEP_MONTH']=df['PLANNED_DEP_DATETIME'].dt.month
df['DEP_DOW']=df['PLANNED_DEP_DATETIME'].dt.dayofweek
df=df.drop(['PLANNED_DEP_DATETIME'],axis=1)

In [9]:
hubs=['CLT','ORD','DFW','LAX','MIA','JFK','LGA','PHL','PHX','DCA']
is_origin_hub=df['ORIGIN'].isin(hubs)

is_dest_hub=df['DEST'].isin(hubs) 

In [11]:
df['origin_hub_new']=np.where(df['ORIGIN'].isin(is_origin_hub),1,0)

In [13]:
df['dest_hub_new']=np.where(df['ORIGIN'].isin(is_dest_hub),1,0)

In [17]:
df=df.drop(['FL_NUM','ORIGIN','DEST'],axis=1)

In [24]:
y=df['CARRIER_DELAY'].map(lambda x: 1 if x>15 else 0 )
X=df.drop(['CARRIER_DELAY'],axis=1)

In [27]:
y.head(20)

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     1
9     0
10    0
11    0
12    0
13    0
14    0
15    0
16    0
17    0
18    1
19    1
Name: CARRIER_DELAY, dtype: int64

In [28]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.15,random_state=8)

In [29]:
corr=df.corr()

In [30]:
print(corr)

                     CRS_DEP_TIME  DEP_TIME  DEP_DELAY  DEP_AFPH  DEP_RFPH  \
CRS_DEP_TIME             1.000000  0.963752   0.102027  0.181101 -0.010316   
DEP_TIME                 0.963752  1.000000   0.145238  0.188205 -0.093217   
DEP_DELAY                0.102027  0.145238   1.000000 -0.002624  0.105752   
DEP_AFPH                 0.181101  0.188205  -0.002624  1.000000 -0.019647   
DEP_RFPH                -0.010316 -0.093217   0.105752 -0.019647  1.000000   
TAXI_OUT                 0.020949  0.029965   0.052265  0.088614 -0.033278   
WHEELS_OFF               0.932651  0.964042   0.136996  0.198437 -0.100034   
CRS_ELAPSED_TIME        -0.005419 -0.007768   0.009739 -0.020587 -0.013970   
PCT_ELAPSED_TIME        -0.002711  0.004154   0.026420  0.053385 -0.025892   
DISTANCE                 0.004192 -0.000051   0.008323  0.002714 -0.009371   
CRS_ARR_TIME             0.591312  0.595166   0.090329  0.181524 -0.059224   
ARR_AFPH                -0.292128 -0.282426  -0.061028 -0.353532

In [32]:
corr['DEP_TIME'].sort_values(ascending=False)

DEP_TIME               1.000000
WHEELS_OFF             0.964042
CRS_DEP_TIME           0.963752
CRS_ARR_TIME           0.595166
DEP_AFPH               0.188205
DEP_DELAY              0.145238
ARR_DELAY              0.136064
LATE_AIRCRAFT_DELAY    0.131146
ARR_RFPH               0.096199
NAS_DELAY              0.056188
WEATHER_DELAY          0.039697
CARRIER_DELAY          0.030941
TAXI_OUT               0.029965
DEP_DOW                0.009133
PCT_ELAPSED_TIME       0.004154
SECURITY_DELAY         0.002779
DISTANCE              -0.000051
DEP_MONTH             -0.000744
CRS_ELAPSED_TIME      -0.007768
DEP_RFPH              -0.093217
ARR_AFPH              -0.282426
origin_hub_new              NaN
dest_hub_new                NaN
Name: DEP_TIME, dtype: float64